# **Estudo Independente 03** - Coleta de Dados utilizando APIs ou Web Scraping

14-11-2026

## 1. Origem dos Dados
* **Fonte:** [Portal de Dados Abertos de Porto Alegre](https://dadosabertos.poa.br/) (Secretaria Municipal de Planejamento e Gestão).
* **Conjunto de Dados:** Postos de Confiança (Cargos em Comissão, Funções Gratificadas e Subsídios).
* **Método de Coleta:** Consumo da **API pública CKAN** (`package_show`), que retorna um JSON com os metadados do conjunto e localiza automaticamente o link atualizado do arquivo de dados.

## 1. Consumo da API Pública e Coleta

In [1]:
import requests
import pandas as pd

# 1. Consulta à API pública do Portal de Dados Abertos (padrão CKAN)
url_api = "https://dadosabertos.poa.br/api/3/action/package_show"
resposta = requests.get(url_api, params={"id": "postos-de-confianca"}, timeout=30)
resposta.raise_for_status()  # Garante que a requisição teve sucesso (Status 200)

# 2. Extração dos metadados retornados em formato JSON
metadados = resposta.json()["result"]
print("Título do Conjunto:", metadados["title"])
print("Organização:", metadados["organization"]["title"])
print("Licença:", metadados["license_title"])

# 3. Localização automática do link do arquivo CSV dentro dos recursos da API
url_csv = None
for recurso in metadados["resources"]:
    if recurso["format"].upper() == "CSV":
        url_csv = recurso["url"]
        break

print("URL do CSV encontrada:", url_csv)

# 4. Leitura dos dados brutos (separados por ponto e vírgula)
df = pd.read_csv(url_csv, sep=";")
print(f"\nDimensões da base coletada: {df.shape[0]} linhas e {df.shape[1]} colunas.")
df.head()

Título do Conjunto: Postos de Confiança
Organização: Secretaria Municipal de Planejamento e Gestão
Licença: Creative Commons Attribution
URL do CSV encontrada: https://dadosabertos.poa.br/dataset/bc544f64-5e86-4a60-8dcb-527c06a24021/resource/bd7dcbce-7800-43bc-a80c-b4a2c012ac76/download/postos_confianca.csv

Dimensões da base coletada: 2853 linhas e 8 colunas.


,data_extracao,secretaria,descricao,numero_vaga,nivel,lei,dtini,dtfim
0,2026-10-01 00:00:00,SMC,SECRETARIO MUNICIPAL,1000980,SUBSIDIO,LE,2005-07-01 00:00:00,NaN
1,2026-10-01 00:00:00,SMSEG,SECRETARIO MUNICIPAL,1001905,SUBSIDIO,LE,2005-07-01 00:00:00,NaN
2,2026-10-01 00:00:00,SMS,SECRETARIO MUNICIPAL,1001172,SUBSIDIO,LE,2005-07-01 00:00:00,NaN
3,2026-10-01 00:00:00,GP,PREFEITO,1000001,SUBSIDIO,LOM,2005-07-01 00:00:00,NaN
4,2026-10-01 00:00:00,GP,VICE-PREFEITO,1000002,SUBSIDIO,LOM,2005-07-01 00:00:00,NaN


## 3. Padronização e Estruturação dos Dados
Antes de armazenar os dados para uso nas próximas etapas analíticas, realizamos a higienização estrutural:
1. Conversão das colunas de datas (que vêm como texto no CSV) para o tipo `datetime`.
2. Limpeza de espaços extras (`.str.strip()`) e padronização de caixa (`.str.upper()` / `.str.title()`).
3. Remoção de eventuais registros duplicados.

In [2]:
# 1. Conversão de tipos: Texto -> Data (valores vazios em dtfim tornam-se NaT)
for col in ["data_extracao", "dtini", "dtfim"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 2. Padronização das colunas de texto
df["secretaria"] = df["secretaria"].str.strip().str.upper()
df["descricao"] = df["descricao"].str.strip().str.title()
df["nivel"] = df["nivel"].str.strip().str.upper()
df["lei"] = df["lei"].str.strip()

# 3. Remoção de duplicatas exatas
df = df.drop_duplicates().reset_index(drop=True)

# 4. Verificação da estrutura final (tipos de dados e preenchimento)
print("--- Estrutura do DataFrame Após Padronização ---")
df.info()

print("\n--- Resumo da Coleta ---")
print("Data da extração na origem:", df["data_extracao"].max().date())
print("Total de vagas estruturadas:", len(df))
print("Secretarias mapeadas:", df["secretaria"].nunique())
df.head()

--- Estrutura do DataFrame Após Padronização ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2853 entries, 0 to 2852
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   data_extracao  2853 non-null   datetime64[ns]
 1   secretaria     2853 non-null   object        
 2   descricao      2853 non-null   object        
 3   numero_vaga    2853 non-null   int64         
 4   nivel          2853 non-null   object        
 5   lei            2853 non-null   object        
 6   dtini          2853 non-null   datetime64[ns]
 7   dtfim          1 non-null      datetime64[ns]
dtypes: datetime64[ns](3), int64(1), object(4)
memory usage: 178.4+ KB

--- Resumo da Coleta ---
Data da extração na origem: 2026-10-01
Total de vagas estruturadas: 2853
Secretarias mapeadas: 21


,data_extracao,secretaria,descricao,numero_vaga,nivel,lei,dtini,dtfim
0,2026-10-01,SMC,Secretario Municipal,1000980,SUBSIDIO,LE,2005-07-01,NaT
1,2026-10-01,SMSEG,Secretario Municipal,1001905,SUBSIDIO,LE,2005-07-01,NaT
2,2026-10-01,SMS,Secretario Municipal,1001172,SUBSIDIO,LE,2005-07-01,NaT
3,2026-10-01,GP,Prefeito,1000001,SUBSIDIO,LOM,2005-07-01,NaT
4,2026-10-01,GP,Vice-Prefeito,1000002,SUBSIDIO,LOM,2005-07-01,NaT


## 4. Armazenamento em Formato Estruturado (CSV e JSON)
Exportação da base estruturada em dois padrões abertos (`CSV` e `JSON`) para utilização direta na etapa seguinte de Análise Exploratória e Visualização de Dados.

In [5]:
from google.colab import files

# 1. Salva em CSV estruturado (sem a coluna de índice numérico)
df.to_csv("dados_ei_03_postos_confianca_estruturado.csv", index=False, encoding="utf-8")

print("Arquivos gerados com sucesso!")

# 3. Faz o download para o computador
files.download("dados_ei_03_postos_confianca_estruturado.csv")

Arquivos gerados com sucesso!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>